
### MODULE 2 LAB 4
### Text Representation


---

## **Section 1: Understanding NLP tools**

In this lab we will be using KNN on a real world NLP application i.e. is text classification. But first look at some NLP techniques for text classification and tools that we use when we want to use python for NLP.

## Section 1.2: Data Cleaning and Preprocessing step

Raw text must be processed and converted into a form so that it is suitable to use with various machine-learning algorithms.  
In case of text, there are lots of things that need to be taken into account.  


1.   Removing numbers from the text
2.   Handling capitalization and punctuation.
3.   Stemming and Lemmatizing text.  

And most importantly, one can't just use words or images directly in algorithms; they need to be converted into vectors- a form that algorithms can understand.



### **NLTK**
NLTK (or Natural Language Tool Kit) is a commonly used library for processing text. We will use this tool in this lab. Lets first install it.


In [ ]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')
nltk.download('omw-1.4')

In [ ]:
import re
import numpy
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.stem import SnowballStemmer
from nltk.tokenize import word_tokenize
from nltk import pos_tag
from nltk.corpus import wordnet
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from bs4 import BeautifulSoup

def cleanText(text, lemmatize, stemmer):
    """Method for cleaning text from train and test data. Removes numbers, punctuation, and capitalization. Stems or lemmatizes text."""

    if isinstance(text, float):
        text = str(text)
    if isinstance(text, numpy.int64):
        text = str(text)
    try:
        text = text.decode()
    except AttributeError:
        pass

    soup = BeautifulSoup(text, "lxml")
    text = soup.get_text()
    text = re.sub(r"[^A-Za-z]", " ", text)
    text = text.lower()


    if lemmatize:
        wordnet_lemmatizer = WordNetLemmatizer()

        def get_tag(tag):
            if tag.startswith('J'):
                return wordnet.ADJ
            elif tag.startswith('V'):
                return wordnet.VERB
            elif tag.startswith('N'):
                return wordnet.NOUN
            elif tag.startswith('R'):
                return wordnet.ADV
            else:
                return ''

        text_result = []
        tokens = word_tokenize(text)  # Generate list of tokens
        tagged = pos_tag(tokens)
        for t in tagged:
            try:
                text_result.append(wordnet_lemmatizer.lemmatize(t[0], get_tag(t[1][:2])))
            except:
                text_result.append(wordnet_lemmatizer.lemmatize(t[0]))
        return text_result

    if stemmer:
        text_result = []
        tokens = word_tokenize(text)
        snowball_stemmer = SnowballStemmer('english')
        for t in tokens:
            text_result.append(snowball_stemmer.stem(t))
        return text_result

In [ ]:
sample_text = "Troubling"
sample_text_result = cleanText(sample_text, lemmatize=False, stemmer=True)
sample_text_result = " ".join(str(x) for x in sample_text_result)
print(sample_text)
print(sample_text_result)
sample_text_result = cleanText(sample_text, lemmatize=True, stemmer=False)
sample_text_result = " ".join(str(x) for x in sample_text_result)
print(sample_text_result)

## Section 1.2: BAG OF WORDS

A bag-of-words model, or BoW for short, is a way of extracting features from text for use in modeling, such as with machine learning algorithms.

The approach is very simple and flexible, and can be used in many ways for extracting features from documents.

A bag-of-words is a representation of text that describes the occurrence of words within a document.
It is called a “bag” of words, because any information about the order or structure of words in the document is discarded. The model is only concerned with whether known words occur in the document, not where in the document.

In [ ]:
5*12

In [ ]:
# Functions to convert document(s) to a list of words, with the option of removing stopwords. Returns document-term matrix.

def createBagOfWords(train, test, remove_stopwords, lemmatize, stemmer):
    if remove_stopwords:
        vectorizer = CountVectorizer(analyzer='word', input='content', stop_words=stopwords.words('english'))
    else:
        vectorizer = CountVectorizer(analyzer='word', input='content')

    clean_train = []
    for paragraph in train:
        paragraph_result = cleanText(paragraph, lemmatize, stemmer)
        paragraph = " ".join(str(x) for x in paragraph_result)
        clean_train.append(paragraph)

    clean_test = []
    for paragraph in test:
        paragraph_result = cleanText(paragraph, lemmatize, stemmer)
        paragraph = " ".join(str(x) for x in paragraph_result)
        clean_test.append(paragraph)

    bag_of_words_train = vectorizer.fit_transform(clean_train).toarray()
    bag_of_words_test = vectorizer.transform(clean_test).toarray()
    return bag_of_words_train, bag_of_words_test


In [ ]:
# Visualize how Bag of Words works with a simple example
from sklearn.feature_extraction.text import CountVectorizer

sample_sentences = [
    "I love this movie",
    "This movie is great",
    "I hate this movie"
]

vectorizer = CountVectorizer()
bow_matrix = vectorizer.fit_transform(sample_sentences)

print("Vocabulary (unique words):", vectorizer.get_feature_names_out())
print("\nBag of Words Matrix:")
print(bow_matrix.toarray())
print("\nEach row represents a sentence, each column represents a word count")

## Section 1.3: TF-IDF
TF-IDF technique is used to find meaning of sentences consisting of words and cancels out the incapabilities of Bag of Words technique which is good for text classification or for helping a machine read words in numbers.

The number of times a term occurs in a document is called its Term frequency (TF).

 Document frequency is the number of documents in which the word is present.  Inverse DF (IDF) is the inverse of the document frequency which measures the informativeness of term *t*.




In [ ]:
def createTFIDF(train, test, remove_stopwords, lemmatize, stemmer):
    if remove_stopwords:
        vectorizer = TfidfVectorizer(analyzer='word', input='content', stop_words=stopwords.words('english'))
    else:
        vectorizer =  TfidfVectorizer(analyzer='word', input='content')

    clean_train = []
    for paragraph in train:
        paragraph_result = cleanText(paragraph, lemmatize, stemmer)
        paragraph = " ".join(str(x) for x in paragraph_result)
        clean_train.append(paragraph)

    clean_test = []
    for paragraph in test:
        paragraph_result = cleanText(paragraph, lemmatize, stemmer)
        paragraph = " ".join(str(x) for x in paragraph_result)
        clean_test.append(paragraph)

    tfidf_train = vectorizer.fit_transform(clean_train).toarray()
    tfidf_test = vectorizer.transform(clean_test).toarray()
    return tfidf_train, tfidf_test

In [ ]:
# Compare BoW vs TF-IDF on the same example
from sklearn.feature_extraction.text import TfidfVectorizer

sample_sentences = [
    "I love this movie",
    "This movie is great",
    "I hate this movie"
]

tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(sample_sentences)

print("TF-IDF Matrix:")
print(tfidf_matrix.toarray())
print("\nNotice: Common words like 'movie' have lower scores")
print("Unique words like 'love', 'hate', 'great' have higher scores")

# **Section 2: UNDERSTANDING THE DATA : A REVIEWS DATASET**

Sentiment analysis is the interpretation and classification of emotions (such as positive, negative and neutral) within text data using text analysis techniques.  
Given below is a dataset consisting of reviews along with sentiment class (positive or negative).

In [ ]:
# Upload reviews.csv in Google Colab.
# In local Jupyter, place reviews.csv in the notebook's working directory.
try:
    from google.colab import files
    uploaded = files.upload()
except ImportError:
    print("Local Jupyter detected. Please place reviews.csv in the current working directory.")

In [ ]:
import pandas as pd
from pathlib import Path

reviews_path = Path("reviews.csv")
if not reviews_path.exists():
    raise FileNotFoundError(
        "reviews.csv was not found. Upload the Reviews CSV in the previous cell "
        "or place reviews.csv beside this notebook."
    )

df = pd.read_csv(reviews_path)
required_columns = {"sentence", "sentiment"}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(
        f"reviews.csv must contain columns {sorted(required_columns)}. "
        f"Missing: {sorted(missing_columns)}"
    )

df = df[["sentence", "sentiment"]].dropna().copy()
df["sentence"] = df["sentence"].astype(str)
df["sentiment"] = df["sentiment"].astype(str).str.strip()
df = df[df["sentence"].str.strip().ne("")]
df = df.reset_index(drop=True)

print("Reviews dataset loaded successfully.")
print("Shape:", df.shape)
df.head()

In [ ]:
df = df.dropna()

In [ ]:
df

In [ ]:
df.to_csv('reviews.csv', index=False)

In [ ]:
# Explore the dataset to understand its structure
print("Dataset Shape:", df.shape)
print("\nFirst few rows:")
print(df.head())

print("\nClass Distribution:")
print(df['sentiment'].value_counts())

print("\nSample Reviews:")
for i in range(3):
    print(f"\nReview {i+1}: {df['sentence'].iloc[i]}")
    print(f"Sentiment: {df['sentiment'].iloc[i]}")

# **Section 3: KNN MODEL**

Given below are two KNN models; in the first case we are using Bag-of-Words and in the second case we are using TF-IDF.
Note the different metrics and parameters used in each.

In [ ]:
from sklearn import metrics, neighbors
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict

## TASK - 1: Tweak the models below and see results with different parameters and distance metrics.

def bow_knn():
    """Method for determining nearest neighbors using bag-of-words and K-Nearest Neighbor algorithm"""

    training_data = pd.read_csv('reviews.csv')
    X_train, X_test, y_train, y_test = train_test_split(training_data["sentence"], training_data["sentiment"], test_size=0.2, random_state=5)
    X_train, X_test = createBagOfWords(X_train, X_test, remove_stopwords=True, lemmatize=True, stemmer=False)
    # print(X_train)
    knn = neighbors.KNeighborsClassifier(n_neighbors=5, weights='uniform', algorithm='auto', leaf_size=30, p=2, metric='euclidean', metric_params=None, n_jobs=1)

    knn.fit(X_train, y_train)
    predicted = knn.predict(X_test)
    acc = metrics.accuracy_score(y_test, predicted)
    print('KNN with BOW accuracy = ' + str(acc * 100) + '%')

    scores = cross_val_score(knn, X_train, y_train, cv=3)
    print("Cross Validation Accuracy: %0.2f" % (scores.mean()))
    print(scores)
    print('\n')
    return predicted, y_test


def tfidf_knn():
    """Method for determining nearest neighbors using tf-idf and K-Nearest Neighbor algorithm"""

    training_data = pd.read_csv('reviews.csv')
    X_train, X_test, y_train, y_test = train_test_split(training_data["sentence"], training_data["sentiment"],
                                                        test_size=0.2, random_state=5)
    X_train, X_test = createTFIDF(X_train, X_test, remove_stopwords=True, lemmatize=True, stemmer=False)
    # print(X_train)
    knn = neighbors.KNeighborsClassifier(n_neighbors=5, weights='distance', algorithm='brute', leaf_size=30, p=2,
                                         metric='cosine', metric_params=None, n_jobs=1)

    knn.fit(X_train, y_train)
    predicted = knn.predict(X_test)
    acc = metrics.accuracy_score(y_test, predicted)
    print('KNN with TFIDF accuracy = ' + str(acc * 100) + '%')

    scores = cross_val_score(knn, X_train, y_train, cv=3)
    print("Cross Validation Accuracy: %0.2f" % (scores.mean()))
    print(scores)
    return predicted, y_test

Note: Cross-validation will be discussed in detail in the upcoming lab session.

In [ ]:
## KNN accuracy after using BoW
predicted, y_test = bow_knn()

In [ ]:
## KNN accuracy after using TFIDF
predicted, y_test = tfidf_knn()

In [ ]:
# Visualize model performance with confusion matrix
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# Create confusion matrix
cm = confusion_matrix(y_test, predicted)

# Plot
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
plt.title('Confusion Matrix - KNN with TF-IDF')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

print("\nDetailed Classification Report:")
print(classification_report(y_test, predicted, target_names=['Negative', 'Positive']))

## TASK 1 – Completed: Compare and tune KNN for the Reviews dataset

The starter exercise asks you to change KNN parameters and distance metrics. The cell below performs a reproducible comparison for both text representations:

- Bag of Words (CountVectorizer) with Euclidean distance.
- Bag of Words with cosine distance.
- TF-IDF with cosine distance.
- TF-IDF with Euclidean distance.

It tunes the number of neighbours (`k`) and the voting weights (`uniform` or `distance`). The vectorizer is inside a Pipeline so vocabulary and IDF values are learned from each training fold only. This avoids data leakage during cross-validation.

**Important:** Results depend on the exact `reviews.csv` supplied with the course. The notebook prints the actual scores when you run it; no accuracy values are assumed here.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import pandas as pd
import numpy as np

# Use the cleaned reviews dataframe created in Section 2.
X = df["sentence"].astype(str)
y = df["sentiment"].astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=5, stratify=y
)

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=5)

review_experiments = {
    "BoW + Euclidean": {
        "vectorizer": CountVectorizer(stop_words="english", ngram_range=(1, 1)),
        "metric": "euclidean"
    },
    "BoW + Cosine": {
        "vectorizer": CountVectorizer(stop_words="english", ngram_range=(1, 1)),
        "metric": "cosine"
    },
    "TF-IDF + Euclidean": {
        "vectorizer": TfidfVectorizer(stop_words="english", ngram_range=(1, 1), min_df=1),
        "metric": "euclidean"
    },
    "TF-IDF + Cosine": {
        "vectorizer": TfidfVectorizer(stop_words="english", ngram_range=(1, 1), min_df=1),
        "metric": "cosine"
    }
}

review_results = []
review_best_models = {}

for experiment_name, settings in review_experiments.items():
    pipe = Pipeline([
        ("vectorizer", settings["vectorizer"]),
        ("knn", KNeighborsClassifier(algorithm="brute", metric=settings["metric"]))
    ])

    param_grid = {
        "knn__n_neighbors": [3, 5, 7],
        "knn__weights": ["uniform", "distance"]
    }

    search = GridSearchCV(
        pipe,
        param_grid=param_grid,
        scoring="f1_macro",
        cv=cv,
        n_jobs=1,
        refit=True
    )
    search.fit(X_train, y_train)
    predictions = search.predict(X_test)

    review_best_models[experiment_name] = search.best_estimator_
    review_results.append({
        "Experiment": experiment_name,
        "Best parameters": search.best_params_,
        "CV macro-F1": search.best_score_,
        "Test accuracy": accuracy_score(y_test, predictions),
        "Test macro-F1": f1_score(y_test, predictions, average="macro")
    })

review_results_df = pd.DataFrame(review_results).sort_values(
    "Test macro-F1", ascending=False
).reset_index(drop=True)

print("TASK 1 – Reviews: model comparison")
display(review_results_df)

best_review_name = review_results_df.loc[0, "Experiment"]
best_review_model = review_best_models[best_review_name]
best_review_predictions = best_review_model.predict(X_test)

print("Best experiment:", best_review_name)
print("\nClassification report:")
print(classification_report(y_test, best_review_predictions, zero_division=0))
print("Confusion matrix (rows = actual, columns = predicted):")
print(confusion_matrix(y_test, best_review_predictions, labels=best_review_model.classes_))

# Section 4: SPAM TEXT DATASET
Now let's use what we've learnt to classify texts as spam or not spam.

In [ ]:
# Upload spam.csv in Google Colab.
# In local Jupyter, place spam.csv in the notebook's working directory.
try:
    from google.colab import files
    uploaded = files.upload()
except ImportError:
    print("Local Jupyter detected. Please place spam.csv in the current working directory.")

In [ ]:
import pandas as pd
from pathlib import Path

spam_path = Path("spam.csv")
if not spam_path.exists():
    raise FileNotFoundError(
        "spam.csv was not found. Upload spam.csv in the previous cell "
        "or place spam.csv beside this notebook."
    )

spam_df = pd.read_csv(spam_path, encoding="latin-1")

# The UCI SMS Spam Collection commonly has v1 (label), v2 (message),
# plus three unused columns. Keep the first two columns.
if spam_df.shape[1] < 2:
    raise ValueError("spam.csv must contain at least two columns: label and message.")

spam_df = spam_df.iloc[:, :2].copy()
spam_df.columns = ["Category", "Message"]
spam_df = spam_df.dropna()
spam_df["Category"] = spam_df["Category"].astype(str).str.strip().str.lower()
spam_df["Message"] = spam_df["Message"].astype(str)
spam_df = spam_df[spam_df["Category"].isin(["ham", "spam"])].copy()
spam_df["Category"] = spam_df["Category"].map({"ham": 0, "spam": 1})
spam_df = spam_df.reset_index(drop=True)

print("Spam dataset loaded successfully.")
print("Shape:", spam_df.shape)
print("Class distribution (0 = ham, 1 = spam):")
print(spam_df["Category"].value_counts())
spam_df.head()

In [ ]:
# Numeric target: ham = 0, spam = 1
df = spam_df.copy()
df.head()

In [ ]:
df.head(5)

In [ ]:
print("Number of SMS messages:", len(df))

In [ ]:
from sklearn import metrics, neighbors
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict

## TASK - 2: Tweak the models below and see results with different parameters

def bow_knn():
    """Method for determining nearest neighbors using bag-of-words and K-Nearest Neighbors."""

    # Load data with proper encoding
    training_data = pd.read_csv('spam.csv', encoding='latin-1')

    # Clean the data - keep only first 2 columns if there are more
    if training_data.shape[1] > 2:
        training_data = training_data.iloc[:, :2]
    training_data.columns = ['Category', 'Message']
    training_data = training_data.dropna()

    # Map categories to numeric
    training_data['Category'] = training_data['Category'].map({'ham': 0, 'spam': 1})

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        training_data["Message"],
        training_data["Category"],
        test_size=0.2,
        random_state=0
    )

    # Create Bag of Words features
    X_train, X_test = createBagOfWords(X_train, X_test, remove_stopwords=True,
                                        lemmatize=True, stemmer=False)

    # Train KNN model
    knn = neighbors.KNeighborsClassifier(n_neighbors=5, weights='uniform',
                                          metric='euclidean')
    knn.fit(X_train, y_train)

    # Make predictions
    predicted = knn.predict(X_test)

    # Calculate accuracy
    acc = metrics.accuracy_score(y_test, predicted)
    print('KNN with BOW accuracy = ' + str(acc * 100) + '%')

    # Cross-validation
    scores = cross_val_score(knn, X_train, y_train, cv=3)
    print("Cross Validation Accuracy: %0.2f" % (scores.mean()))
    print(scores)
    print('\n')

    return predicted, y_test


def tfidf_knn():
    """Method for determining nearest neighbors using tf-idf and K-Nearest Neighbors."""

    # Load data with proper encoding
    training_data = pd.read_csv('spam.csv', encoding='latin-1')

    # Clean the data - keep only first 2 columns if there are more
    if training_data.shape[1] > 2:
        training_data = training_data.iloc[:, :2]
    training_data.columns = ['Category', 'Message']
    training_data = training_data.dropna()

    # Map categories to numeric
    training_data['Category'] = training_data['Category'].map({'ham': 0, 'spam': 1})

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        training_data["Message"],
        training_data["Category"],
        test_size=0.2,
        random_state=0
    )

    # Create TF-IDF features
    X_train, X_test = createTFIDF(X_train, X_test, remove_stopwords=True,
                                   lemmatize=True, stemmer=False)

    # Train KNN model
    knn = neighbors.KNeighborsClassifier(n_neighbors=5, weights='distance',
                                          metric='cosine', metric_params=None)
    knn.fit(X_train, y_train)

    # Make predictions
    predicted = knn.predict(X_test)

    # Calculate accuracy
    acc = metrics.accuracy_score(y_test, predicted)
    print('KNN with TFIDF accuracy = ' + str(acc * 100) + '%')

    # Cross-validation
    scores = cross_val_score(knn, X_train, y_train, cv=3)
    print("Cross Validation Accuracy: %0.2f" % (scores.mean()))
    print(scores)

    return predicted, y_test

In [ ]:
# This cell may take some time to run
predicted, y_test = bow_knn()

In [ ]:
# This cell may take some time to run
predicted, y_test = tfidf_knn()

## TASK 2 – Completed: Compare and tune KNN for SMS Spam detection

This experiment repeats the comparison on the SMS dataset. It compares BoW and TF-IDF, tunes `k` and neighbour weighting, and evaluates the final test set.

For spam detection, accuracy alone can be misleading because ham messages are usually much more common than spam. Therefore, inspect **precision, recall and F1-score for the spam class (label 1)** as well as the confusion matrix.

- **Spam precision:** Of messages predicted as spam, how many really are spam?
- **Spam recall:** Of all actual spam messages, how many did the model catch?
- **Spam F1-score:** Harmonic mean of spam precision and recall.

In [ ]:
# Use df prepared in Section 4: Category 0 = ham, Category 1 = spam.
X_spam = df["Message"].astype(str)
y_spam = df["Category"].astype(int)

X_spam_train, X_spam_test, y_spam_train, y_spam_test = train_test_split(
    X_spam, y_spam, test_size=0.20, random_state=0, stratify=y_spam
)

spam_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=0)

spam_experiments = {
    "BoW + Euclidean": {
        "vectorizer": CountVectorizer(stop_words="english", ngram_range=(1, 1)),
        "metric": "euclidean"
    },
    "BoW + Cosine": {
        "vectorizer": CountVectorizer(stop_words="english", ngram_range=(1, 1)),
        "metric": "cosine"
    },
    "TF-IDF + Euclidean": {
        "vectorizer": TfidfVectorizer(stop_words="english", ngram_range=(1, 1)),
        "metric": "euclidean"
    },
    "TF-IDF + Cosine": {
        "vectorizer": TfidfVectorizer(stop_words="english", ngram_range=(1, 1)),
        "metric": "cosine"
    }
}

spam_results = []
spam_best_models = {}

for experiment_name, settings in spam_experiments.items():
    pipe = Pipeline([
        ("vectorizer", settings["vectorizer"]),
        ("knn", KNeighborsClassifier(algorithm="brute", metric=settings["metric"]))
    ])

    param_grid = {
        "knn__n_neighbors": [3, 5, 7],
        "knn__weights": ["uniform", "distance"]
    }

    search = GridSearchCV(
        pipe,
        param_grid=param_grid,
        scoring="f1",
        cv=spam_cv,
        n_jobs=1,
        refit=True
    )
    search.fit(X_spam_train, y_spam_train)
    predictions = search.predict(X_spam_test)

    spam_best_models[experiment_name] = search.best_estimator_
    spam_results.append({
        "Experiment": experiment_name,
        "Best parameters": search.best_params_,
        "CV spam F1": search.best_score_,
        "Test accuracy": accuracy_score(y_spam_test, predictions),
        "Test spam precision": precision_score(y_spam_test, predictions, zero_division=0),
        "Test spam recall": recall_score(y_spam_test, predictions, zero_division=0),
        "Test spam F1": f1_score(y_spam_test, predictions, zero_division=0)
    })

spam_results_df = pd.DataFrame(spam_results).sort_values(
    "Test spam F1", ascending=False
).reset_index(drop=True)

print("TASK 2 – SMS Spam: model comparison")
display(spam_results_df)

best_spam_name = spam_results_df.loc[0, "Experiment"]
best_spam_model = spam_best_models[best_spam_name]
best_spam_predictions = best_spam_model.predict(X_spam_test)

print("Best experiment:", best_spam_name)
print("\nClassification report (0 = ham, 1 = spam):")
print(classification_report(
    y_spam_test, best_spam_predictions,
    labels=[0, 1], target_names=["ham", "spam"], zero_division=0
))

cm_spam = confusion_matrix(y_spam_test, best_spam_predictions, labels=[0, 1])
print("Confusion matrix (rows = actual [ham, spam], columns = predicted [ham, spam]):")
print(cm_spam)

### Questions to Think About and Answer
1. Why does the TF-IDF approach generally result in a better accuracy than Bag-of-Words ?
2. Can you think of techniques that are better than both BoW and TF-IDF ?
3. Read about Stemming and Lemmatization from the resources given below. Think about the pros/cons of each.

## LAB QUESTIONS – COMPLETED ANSWERS

### 1. Why can TF-IDF result in better accuracy than Bag of Words?

Bag of Words represents a document using the raw count of each word. A word that appears many times receives a large count, even if that word is common across almost every document.

TF-IDF gives a word a weight based on two ideas:

- **Term Frequency (TF):** how often the word occurs in the current document.
- **Inverse Document Frequency (IDF):** how uncommon the word is across the collection of documents.

A word that appears in many documents receives a lower IDF weight. A word that is frequent in one document but uncommon across the dataset can receive a higher weight. This can make informative words more influential and reduce the influence of very common words.

**Example:** In movie reviews, words such as “movie” and “film” may appear in many reviews and may not distinguish positive from negative sentiment. Words such as “excellent”, “boring” or “terrible” may be more informative. TF-IDF can give common words less weight.

**Important qualification:** TF-IDF does not always achieve better accuracy. The result depends on the dataset, preprocessing, classifier, distance metric, hyperparameters and evaluation split. The experiment in Task 1 should be used to determine which representation works better on the supplied reviews dataset.

### 2. What techniques can be better than both BoW and TF-IDF?

Several methods can represent meaning and context more effectively:

| Technique | How it works | Main advantage |
|---|---|---|
| Word2Vec | Learns a dense vector for each word from surrounding context | Captures some semantic relationships |
| GloVe | Learns word vectors from global word co-occurrence statistics | Represents semantic similarity |
| FastText | Learns word representations using character n-grams | Can handle rare and unseen word forms better |
| Doc2Vec | Learns a vector representation for a document | Represents whole documents |
| BERT / transformer embeddings | Produces context-sensitive representations | The same word can have different vectors depending on context |
| Sentence Transformers | Produces embeddings for sentences or paragraphs | Useful for semantic similarity and text classification |

These approaches are not automatically better for every task. They may need more computation, data or model selection. For a small dataset, BoW or TF-IDF can be a strong, simple baseline.

### 3. Stemming and Lemmatization – advantages and disadvantages

**Stemming** removes word endings using heuristic rules. It may produce a word stem that is not a valid dictionary word.

Examples:
- “playing” → “play”
- “studies” → “studi” (depending on the stemmer)

Advantages:
- Usually fast.
- Simple to implement.
- Can reduce vocabulary size.
- Useful when related word forms should be grouped.

Disadvantages:
- Can produce non-dictionary stems.
- May remove too much or too little.
- Does not fully understand the word's grammatical role or meaning.

**Lemmatization** converts a word to its dictionary base form (lemma), often using vocabulary and grammatical information.

Examples:
- “running” → “run” (verb)
- “better” → “good” (when correctly interpreted as an adjective)
- “cars” → “car”

Advantages:
- Usually produces meaningful dictionary words.
- Uses grammatical information when POS tagging is provided.
- Often preserves meaning better than stemming.

Disadvantages:
- Usually slower than stemming.
- Requires linguistic resources and sometimes part-of-speech tagging.
- Results depend on language and context.

**Conclusion:** Stemming is generally faster and simpler; lemmatization is generally more linguistically informed. Choose based on the task and compare validation results rather than assuming one is always superior.

## FINAL RUN CHECKLIST

1. Run the NLTK download and import cells in Section 1.
2. Upload the course-provided `reviews.csv` when prompted. It must contain `sentence` and `sentiment` columns.
3. Run the Reviews data exploration and original KNN baseline cells.
4. Run **TASK 1 – Completed** to compare and tune the review classifiers.
5. Upload `spam.csv` when prompted. The standard SMS Spam Collection format is supported (first column = `ham`/`spam`, second column = message).
6. Run the spam preparation and baseline cells.
7. Run **TASK 2 – Completed** to compare and tune the spam classifiers.
8. Read the completed written answers above.

### Notes about execution and results

- The two CSV datasets were not included with the uploaded notebook. Therefore, data-dependent model cells are written to execute after you upload the correct course datasets.
- Accuracy, precision, recall, F1-score and confusion matrices are calculated from the real uploaded data at runtime. This completed notebook does not fabricate results.
- The tuning experiments use a stratified train/test split and stratified cross-validation. The test set is kept separate from model selection.
- If a package is missing in local Jupyter, install the required packages with: `pip install nltk pandas numpy scikit-learn beautifulsoup4 lxml matplotlib seaborn`.
- In Google Colab, the notebook's NLTK download cell downloads the required language resources.

### Useful Resources for further reading
1. Stemming and Lemmatization: https://nlp.stanford.edu/IR-book/html/htmledition/stemming-and-lemmatization-1.html
2. TF-IDF and BoW : https://www.analyticsvidhya.com/blog/2020/02/quick-introduction-bag-of-words-bow-tf-idf/
3. TF-IDF: https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html
